In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Natural gas spot prices

Daily settlement prices ($/MMBtu) for natural gas futures over 25 trading days. A supply disruption in week 3 caused a brief spike.

1. Add `abs_change` via `diff()` and `rel_change` via `pct_change()`. Which day had the largest absolute drop? Which had the largest relative drop? Are they the same day — and why or why not?
2. `np.nanmean` and `np.nanstd` on `rel_change`. What does the std tell you about daily price risk in this contract?
3. Add `ewm_price = gas['price'].ewm(span=6).mean()`. Use `np.abs(gas['price'] - gas['ewm_price'])` with `np.argmax` to find the day prices deviated furthest from their smoothed trend. Is it the spike day itself, or the correction day after?

In [15]:
gas = pd.DataFrame({
    'day':   pd.date_range('2023-09-01', periods=25, freq='B'),
    'price': [3.42, 3.51, 3.48, 3.55, 3.63, 3.58, 3.71, 3.68, 3.75, 3.82,
              3.79, 3.86, 4.91, 4.12, 3.95, 3.88, 3.97, 4.04, 3.99, 4.08,
              4.15, 4.22, 4.18, 4.27, 4.33],
})

# Your code here
gas['abs_change'] = gas['price'].diff()
gas['rel_change'] = gas['price'].pct_change()
print(gas.loc[gas['abs_change'].idxmin(),'day'],'had largest abs drop')
print(gas.loc[gas['rel_change'].idxmin(),'day'],'had largest rel drop')
print('on the same day')


m = np.nanmean(gas['rel_change'])
s = np.nanstd(gas['rel_change'])
print('sdt is',s)
print('change is pretty stable')

gas['ewm_price'] = gas['price'].ewm(span = 6).mean()
gap = np.abs(gas['price']-gas['ewm_price'])
print(gas.iloc[np.argmax(gap)]['day'],'has the largest gap')
print('it is the spike day')

2023-09-20 00:00:00 had largest abs drop
2023-09-20 00:00:00 had largest rel drop
on the same day
sdt is 0.06635948680206319
change is pretty stable
2023-09-19 00:00:00 has the largest gap
it is the spike day


---

## Level 2 — Hospital emergency wait times

Monthly average wait times (minutes) at a regional emergency department over 24 months. The hospital introduced a new triage protocol at month 7.

No sub-questions — write your own analysis.

**Your task:** determine whether the protocol worked. At minimum:
- `diff()` to find the months with the sharpest drops
- `ewm(span=4).mean()` to track the smoothed trend — does it clearly shift at month 7?
- `expanding().mean()` to track the running average over time
- `np.nanpercentile` on the month-over-month changes — what does the distribution of changes reveal?

**End with a one-sentence verdict** as a print statement or comment.

In [24]:
ed = pd.DataFrame({
    'month': pd.date_range('2022-01', periods=24, freq='MS'),
    'wait':  [142, 148, 145, 151, 138, 144,
              128, 131, 124, 119, 122, 115,
              118, 112, 115, 109, 113, 106,
              110, 104, 108, 101, 105,  98],
})

# Your code here

ed['diff'] = ed['wait'].diff()
print('sharpest drop is at: ', ed.loc[ed['wait'].idxmin(),'month'])
ed['ewm'] = ed['wait'].ewm(span = 4).mean()
ed.iloc[7]['ewm'] - ed.iloc[6]['ewm']
print('shifted at month7')
ed['exp'] = ed['wait'].expanding().mean()
pp = np.nanpercentile(ed['diff'],q = [2,50,75])
print('mostly drops in month over month change')


sharpest drop is at:  2023-12-01 00:00:00
shifted at month7
mostly drops in month over month change


---

## Level 3 — Three product lines

Weekly revenue for Electronics, Clothing, and Home over 16 weeks. Each line has a different trajectory.

1. Add `wow_change` via `groupby('line').transform(lambda x: x.diff())`. Which line × week had the sharpest single-week drop? Then compute `np.nanstd` on each line's `wow_change` separately — which line is most volatile week-to-week?
2. Add `ewm_rev` via `groupby('line').transform(lambda x: x.ewm(span=4).mean())` and `cum_avg` via `groupby('line').transform(lambda x: x.expanding().mean())`. At week 16, compare `ewm_rev` vs `cum_avg` for each line. The ewm is recency-weighted: if `ewm > cum_avg`, the line is accelerating; if `ewm < cum_avg`, it's decelerating. Which lines are accelerating and which are slowing?
3. Extract each line's `revenue` as an array. `np.corrcoef` → `np.fill_diagonal(..., -np.inf)` → `np.unravel_index(np.argmax(...), ...)` to identify the most-correlated pair. Does the result make intuitive sense given what you know about each line's trajectory?

In [51]:
products = pd.DataFrame({
    'week': list(pd.date_range('2023-01-02', periods=16, freq='W')) * 3,
    'line': ['Electronics']*16 + ['Clothing']*16 + ['Home']*16,
    'revenue': [
        52000, 54200, 53800, 56100, 58400, 57200, 61000, 63500,
        62800, 65400, 68200, 67100, 71300, 74200, 73100, 77500,   # Electronics
        38000, 41200, 36800, 40500, 37900, 42100, 38400, 41800,
        37200, 40900, 38700, 42200, 38100, 41500, 37800, 40200,   # Clothing
        29500, 28900, 29800, 28600, 29200, 27800, 28500, 27300,
        28100, 26900, 27500, 26200, 27100, 25900, 26600, 25400,   # Home
    ],
})

# Your code here

products['wow_change'] = products.groupby('line')['revenue'].transform(lambda x: x.diff())

ss = np.nanargmin(products['wow_change'])
print(products.iloc[ss]['week'], products.iloc[ss]['line'],'has the sharpest week drop')
linestd = products.groupby('line')['wow_change'].apply(lambda x: np.nanstd(x))
print(linestd.idxmax(),'is th emost volatile')
products['ewm_rev'] = products.groupby('line')['revenue'].transform(lambda x: x.ewm(span = 4).mean())
products['cum_avg'] = products.groupby('line')['revenue'].transform(lambda x: x.expanding().mean())

w16 = products.loc[products['week'] == products['week'].unique()[-1]].copy()


w16['d'] = w16['cum_avg'] - w16['ewm_rev']
print('slowing: ',w16.loc[w16['d']>0,'line'])
print('accelerating: ',w16.loc[w16['d']<0,'line'])

ns = ['Electronics','Clothing','Home']
a0 = np.array(products.loc[products['line'] ==ns[0],'revenue'])
a1 = np.array(products.loc[products['line'] ==ns[1],'revenue'])
a2 = np.array(products.loc[products['line'] ==ns[2],'revenue'])

ca = np.corrcoef([a0,a1,a2])
np.fill_diagonal(ca, -np.inf)
row, col = np.unravel_index(np.argmax(ca), shape = ca.shape)
print('most correlated pair is: ', ns[col], ns[row])

print('it makes intuitive sense because both of them are accelerating at week 16')


2023-03-05 00:00:00 Clothing has the sharpest week drop
Clothing is th emost volatile
slowing:  47    Home
Name: line, dtype: object
accelerating:  15    Electronics
31       Clothing
Name: line, dtype: object
most correlated pair is:  Clothing Electronics
it makes intuitive sense because both of them are accelerating at week 16
